## Define Input Parameters

In [ ]:
from pymongo import MongoClient


city = "Milano"

# Path to the GTFS ZIP file
gtfs_zip_path = "GTFS/gtfs_Milano.zip"

# MongoDB connection
mongo_client = MongoClient("mongodb://localhost:27017/")  # Replace with your MongoDB connection string
db = mongo_client["TransitDatabaseMilano15"]

# Define the date
specific_date = "2024-12-23"

# Define the hexagonal grid edge length in kilometers
gridEdge = 0.25  # in kilometers

# OSRM endpoint
OSRM_BASE_URL = "http://localhost:5000"

MAX_WALKING_TIME = 15 * 60  # 30 minutes in seconds

# Walking speed (in meters per second)
WALKING_SPEED = 1.4  # Average human walking speed (5 km/h)
MAX_WALKING_DISTANCE = WALKING_SPEED *  MAX_WALKING_TIME
aoi_path = r"Boundry\Milano.shp"  # Update with your Area of Interest shapefile path
aoi_POP = r"POP\Milano.shp"
aoi_POI = r"POI\POI_Milano.shp"

MAX_TRAVEL_TIME = 120 * 60 
specific_start_hours = [3, 8, 22]
MAX_WALKING_TIME_STOPS = 5 * 60
MAX_WALKING_DISTANCE_STOP = WALKING_SPEED * MAX_WALKING_TIME_STOPS

# Get the data from GTFS Zip File


In [ ]:
from Library.Library import load_gtfs_data

stops, trips, routes, calendar_dates, calendar = load_gtfs_data(gtfs_zip_path)



# Store GTFS on Database

In [ ]:
from Library.Library import process_and_save_gtfs_to_mongo

# Process and save the loaded GTFS data into MongoDB
process_and_save_gtfs_to_mongo(db, calendar_dates, routes, trips, stops, calendar,aoi_path)

# Store Network on Database

In [ ]:
from Library.Library import process_gtfs_data

#  Run the entire edge creation and saving process
process_gtfs_data(db, gtfs_zip_path, specific_date)

# Create Hexagonal Grid

In [ ]:
from Library.Library import process_hexagonal_grid


process_hexagonal_grid(db, aoi_path, gridEdge)

# Store Population on Database

In [ ]:
from Library.Library import process_grid_population
population_column = "TOT_P_2018"
# Run the function
process_grid_population(db, aoi_POP, population_column)

## Add population to the hexagons

In [ ]:
from Library.Library import process_population_computation

process_population_computation(db)

## Walking time for Points

In [ ]:
from Library.Library import process_points

process_points(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)

## Walking time for Stops

In [ ]:
from Library.Library import process_stops


process_stops(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE_STOP, MAX_WALKING_TIME_STOPS)


## Create Stop ID Index

In [ ]:
from Library.Library import create_stop_ids
create_stop_ids(db, city, output_dir="matrices")

## Store POI from OSM

In [ ]:
from Library.Library import fetch_pois_in_boundary, store_pois_to_mongodb
# Define groups with their respective OSM tag filters.

groups = {
    "Healthcare": [
        {"key": "amenity", "value": "hospital"},
        {"key": "amenity", "value": "pharmacy"},
        {"key": "amenity", "value": "clinic"}
    ],
    "Education": [
        {"key": "amenity", "value": "school"},
        {"key": "amenity", "value": "university"},
        {"key": "amenity", "value": "college"}
    ],
    "Food": [
        {"key": "amenity", "value": "restaurant"},
        {"key": "amenity", "value": "cafe"},
        {"key": "amenity", "value": "bar"}
    ],
    "Retail": [
        {"key": "shop", "value": "supermarket"},
        {"key": "shop", "value": "mall"}
    ],
    "Recreation": [
        {"key": "leisure", "value": "park"},
        {"key": "amenity", "value": "cinema"},
        {"key": "leisure", "value": "sports_centre"}
    ],
    "Public_Service": [
        {"key": "amenity", "value": "police"},
        {"key": "amenity", "value": "fire_station"},
        {"key": "amenity", "value": "post_office"},
        {"key": "aeroway", "value": "aerodrome"}
    ]
}


pois_gdf = fetch_pois_in_boundary(aoi_path, groups)
store_pois_to_mongodb(pois_gdf, db)

## Store Shapefile of POI locally

In [ ]:
from Library.Library  import store_poi_geometry_to_mongodb

# Call the processing function
store_poi_geometry_to_mongodb(db, aoi_POI)

## Find closest stops to each POI

In [ ]:
from Library.Library  import process_poi_reachable_stops

process_poi_reachable_stops(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)


## Find Closest Point to each POI

In [ ]:
from Library.Library import process_poi_reachable_points, remove_unreachable_pois

process_poi_reachable_points(db, OSRM_BASE_URL, MAX_WALKING_DISTANCE, MAX_WALKING_TIME)
remove_unreachable_pois(db)

## Compute Accessibility P2POI | POI2P | P2P

In [ ]:
from Library.Library import process_accessibility_to_pois
process_accessibility_to_pois(db, city, specific_start_hours,MAX_TRAVEL_TIME)

from Library.Library_POI import process_accessibility_from_pois
process_accessibility_from_pois(db, city, specific_start_hours,MAX_TRAVEL_TIME)


from Library.Library_P2P import process_accessibility_average
process_accessibility_average(db, city, specific_start_hours,MAX_TRAVEL_TIME)


## Create Accessibility Map

In [ ]:
thresholds = [30, 45, 60, 90]

from Library.Library import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_P2POI.html",thresholds=thresholds,layer_opacity=0.7)

from Library.Library_POI import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_POI2P.html",thresholds=thresholds,layer_opacity=0.7)

from Library.Library_P2P import visualize_multi_hour_accessibility
visualize_multi_hour_accessibility(points_collection = db['points'], poi_collection = db['POI'],hours = specific_start_hours, html_file = f"Maps/accessibility_map_{city}_P2P.html",thresholds=thresholds,layer_opacity=0.7)
